# 01 — Dataset Inspection

**Goal:** Inspect the AI4Mars data you downloaded and extracted into `data/raw/`.  
Verify the file structure, count images and masks, and confirm that pairing logic works before training.

---

## Why do we inspect before training?

Supervised segmentation requires *correctly paired* image and mask files.  
A mismatch (image A + mask B) would silently produce wrong gradients and a useless model.

Before writing any training code we must verify:
- How are the files named? Do images and masks share stems?
- Are images JPG and masks PNG (as documented), or something else?
- What are the unique pixel values in masks? Do they match the expected class IDs?
- Are the shapes sensible?

> **Prerequisite:** You must have downloaded and extracted the AI4Mars dataset  
> into `data/raw/` before running this notebook.  
> See `00_nasa_api_discovery.ipynb` for download instructions.

In [ ]:
import sys
from pathlib import Path
import numpy as np
from PIL import Image
from collections import Counter

# Add project root to sys.path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_paths import RAW_DATA_DIR, ensure_project_dirs
from src.dataset import find_image_files, find_mask_files, build_pairs_by_stem

ensure_project_dirs()

All project directories are ready.


## Step 1 — Set the Data Root

Point `DATA_ROOT` at the directory that contains extracted AI4Mars files.  
Adjust the path if your extraction produced a different folder structure.

In [ ]:
# Adjust this path if your extraction created a sub-folder inside data/raw/
DATA_ROOT = RAW_DATA_DIR

print(f"DATA_ROOT = {DATA_ROOT}")
print(f"Exists    = {DATA_ROOT.exists()}")

DATA_ROOT = C:\Users\Jacob\AI4Mars\data\raw
Exists    = True


## Step 2 — Count All Files

In [ ]:
all_files = list(DATA_ROOT.rglob("*")) if DATA_ROOT.exists() else []
all_files = [f for f in all_files if f.is_file()]

print(f"Total files under DATA_ROOT: {len(all_files)}")

# Count by extension
ext_counter = Counter(f.suffix.lower() for f in all_files)
print("\nFile counts by extension:")
for ext, count in sorted(ext_counter.items(), key=lambda x: -x[1]):
    print(f"  {ext or '(no ext)':>8s}  {count}")

Total files under DATA_ROOT: 146140

File counts by extension:
      .png  84335
      .jpg  56361
     .jpeg  5433
      .txt  6
       .md  3
     .json  1
      .csv  1


## Step 3 — Find Images and Masks

In [ ]:
image_files = find_image_files(DATA_ROOT)
mask_files  = find_mask_files(DATA_ROOT)

print(f"Image files found : {len(image_files)}")
print(f"Mask files found  : {len(mask_files)}")

Image files found : 146129
Mask files found  : 48142


In [ ]:
# Print a few example paths to understand the naming convention
print("Sample image paths:")
for p in image_files[:5]:
    print(f"  {p.relative_to(DATA_ROOT)}")

print("\nSample mask paths:")
for p in mask_files[:5]:
    print(f"  {p.relative_to(DATA_ROOT)}")

Sample image paths:
  msl-labeled-data-set-v2.1\msl-labeled-data-set-v2.1\images\0003ML0000000370100057E01_DRCL-fh.jpg
  msl-labeled-data-set-v2.1\msl-labeled-data-set-v2.1\images\0003ML0000000370100057E01_DRCL.jpg
  msl-labeled-data-set-v2.1\msl-labeled-data-set-v2.1\images\0003ML0000001500100012D01_DRCL-fh.jpg
  msl-labeled-data-set-v2.1\msl-labeled-data-set-v2.1\images\0003ML0000001500100012D01_DRCL.jpg
  msl-labeled-data-set-v2.1\msl-labeled-data-set-v2.1\images\0003ML0000001640100026D01_DRCL-fh.jpg

Sample mask paths:


## Step 4 — Build Image/Mask Pairs

We match files by stem (filename without extension).  
If the AI4Mars archive uses a different naming convention, you may need to adjust `build_pairs_by_stem` or write a custom pairing function.

In [ ]:
pairs = build_pairs_by_stem(image_files, mask_files)
print(f"Matched pairs : {len(pairs)}")
print(f"Unmatched images: {len(image_files) - len(pairs)}")
print(f"Unmatched masks : {len(mask_files)  - len(pairs)}")

print("\nFirst 3 pairs:")
for img, msk in pairs[:3]:
    print(f"  image : {img.name}")
    print(f"  mask  : {msk.name}\n")

Matched pairs : 0
Unmatched images: 6820
Unmatched masks : 0

First 3 pairs:


## Step 5 — Load One Image and One Mask

Manually verify that the image and mask have sensible shapes and the mask contains the expected class IDs.

In [ ]:
if not pairs:
    print("No pairs found. Ensure DATA_ROOT contains extracted AI4Mars data.")
else:
    img_path, msk_path = pairs[0]

    image = np.array(Image.open(img_path).convert("RGB"))
    mask  = np.array(Image.open(msk_path))

    print(f"Image path  : {img_path}")
    print(f"Image shape : {image.shape}  dtype={image.dtype}")
    print(f"\nMask path   : {msk_path}")
    print(f"Mask shape  : {mask.shape}  dtype={mask.dtype}")
    print(f"Unique IDs  : {np.unique(mask).tolist()}")

    # Sanity-check: expected IDs are 0, 1, 2, 3, and possibly 255
    expected_ids = {0, 1, 2, 3, 255}
    found_ids = set(np.unique(mask).tolist())
    unexpected = found_ids - expected_ids
    if unexpected:
        print(f"\n  WARNING: unexpected class IDs found: {unexpected}")
        print("   Verify the label scheme in the AI4Mars documentation.")
    else:
        print("\n Mask class IDs look as expected.")

No pairs found. Ensure DATA_ROOT contains extracted AI4Mars data.


## Next Steps

If the pairs look correct and the class IDs match expectations:
- Open `02_dataset_viewer.ipynb` to visualise image/mask overlays.

If the pairing logic failed or the class IDs look wrong:
- Inspect the actual directory structure by printing `all_files[:20]`.
- Adjust `find_image_files`, `find_mask_files`, or `build_pairs_by_stem` in `src/dataset.py`.